# 06 - Population Search

Plan section 39: variants, prior-ranked candidates, the matched-budget frontier, and Pareto population selection.

Reproducibility contract (plan section 40): every session begins by
determining its configuration, locating and validating prior state,
and continues from there; it ends by saving metrics and a summary JSON.
Run inside a Kaggle session (or locally) after checking out this
repository, with the repository root as the working directory.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
assert (REPO / 'src' / 'act').is_dir(), (
    f'run this notebook with the repository root as the working directory; '
    f'cwd was {REPO!r}'
)
sys.path.insert(0, str(REPO / 'src'))
print('python', sys.version.split()[0], '| repo', REPO)


# Population search (plan section 39)

Generate and evaluate candidate genomes: one-step module/depth
variants in stable order, prior-informed candidate ranking with
explicit depth penalties, the matched-budget static-vs-adaptive
frontier run, and deterministic Pareto selection over measured
quality, active parameters, and search FLOPs.


In [ ]:
from act.evolution.module_depth import enumerate_module_depth_variants
from act.evolution.hierarchical import SearchPrior, rank_module_candidates
from act.topology import create_genome

genome = create_genome(42, hidden_size=24, vocab_size=32, depth=2,
                       ffn='moe', expert_count=4, active_experts=2)
variants = enumerate_module_depth_variants(genome)
assert len(variants) == 6  # 2 layers x (2 module switches + 1 depth toggle)
assert len({v.architecture_hash for _, v in variants}) == len(variants)
# every variant is a distinct architecture
prior = SearchPrior(module_scores={'attention': 0.9, 'gdn': 0.4},
                    depth_penalty=0.1)
ranked = rank_module_candidates(genome, prior)
assert ranked[0].module == 'attention'  # highest prior wins
print('variants:', [(v.operation, v.target, v.module) for _, v in variants[:3]],
      '...')
print('top-ranked:', ranked[0].layer_id, ranked[0].module,
      round(ranked[0].score, 4))

In [ ]:
# Matched-budget frontier: static MoE control versus adaptive MoE.
from act.evolution.experiment import ExperimentConfig
from act.evolution.frontier import run_frontier

points = run_frontier(
    ExperimentConfig(seed=42, steps=48, arms='fixed,adaptive', device='cpu'),
    seeds=(1,),
)
assert len(points) == 2
for point in points:
    print(f"{point.family}: loss={point.validation_loss:.4f} "
          f"active={point.active_parameter_count} vflops={point.validation_flops}")

In [ ]:
# Deterministic Pareto selection over measured trade-offs: each
# frontier point becomes a candidate over (quality, active params,
# search FLOPs) with its own genome.
from act.evolution.pareto import (
    PopulationCandidate,
    pareto_frontier,
    select_population,
)

point_genomes = (
    genome,
    create_genome(7, hidden_size=24, vocab_size=32, depth=2,
                  ffn='moe', expert_count=4, active_experts=2),
)
candidates = tuple(
    PopulationCandidate(
        genome=point_genome,
        quality=-point.validation_loss,
        active_params=point.active_parameter_count,
        search_flops=point.validation_flops,
    )
    for point, point_genome in zip(points, point_genomes)
)
frontier = pareto_frontier(candidates)
assert frontier, 'the frontier must not be empty'
population = select_population(candidates, size=2)
assert len(population) == 2
print('frontier size:', len(frontier),
      '| selected:', [c.architecture_hash[:8] for c in population])

In [ ]:
import json
from pathlib import Path

record_dir = Path('artifacts/notebook-records')
record_dir.mkdir(parents=True, exist_ok=True)
payload = {'notebook': '06_search',
 'variant_count': len(variants),
 'top_ranked': {'layer': ranked[0].layer_id, 'module': ranked[0].module},
 'frontier_size': len(frontier),
 'selected_hashes': [c.architecture_hash for c in population]}
record = json.dumps(payload, sort_keys=True, indent=2) + '\n'
target = record_dir / '06-search.json'
existing = target.read_text() if target.exists() else None
if existing != record:
    target.write_text(record)
print('record:', target)